# PRODIGY_GA_02 — Image Generation with Pre-trained Models

Task: Utilize pre-trained generative models (Stable Diffusion) to create images from text prompts.

In [ ]:
# --- Step 0: Confirm GPU ---
import torch
print("GPU available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

In [ ]:
# --- Step 1: Install dependencies ---
!pip install diffusers transformers accelerate -q

In [ ]:
# --- Step 2: Load Stable Diffusion pipeline ---
from diffusers import StableDiffusionPipeline

model_id = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
)
pipe = pipe.to("cuda")
pipe.set_progress_bar_config(disable=True)  # cleaner output

In [ ]:
# --- Step 3: Define prompts to generate ---
prompts = [
    "a futuristic city skyline at sunset, digital art",
    "a photorealistic portrait of an old fisherman, dramatic lighting",
    "a cozy cabin in a snowy forest, oil painting style",
    "an astronaut riding a horse on mars, highly detailed",
]

In [ ]:
# --- Step 4: Generate one image per prompt (default settings) ---
import os
os.makedirs("outputs", exist_ok=True)

generated_images = {}
for i, prompt in enumerate(prompts):
    print(f"Generating image {i+1}/{len(prompts)}: {prompt}")
    image = pipe(prompt, num_inference_steps=50, guidance_scale=7.5).images[0]
    filename = f"outputs/image_{i+1}.png"
    image.save(filename)
    generated_images[prompt] = filename
    display(image)

## Parameter Exploration

Understanding how generation parameters affect output is core to this task.

In [ ]:
# --- Step 5: Parameter exploration — guidance scale comparison ---
# guidance_scale controls how strictly the model follows the prompt
# (low = more creative/random, high = more literal/prompt-adherent)
test_prompt = "a futuristic city skyline at sunset, digital art"
guidance_scales = [3, 7.5, 15]

os.makedirs("outputs/guidance_comparison", exist_ok=True)
for gs in guidance_scales:
    print(f"Generating with guidance_scale={gs}")
    image = pipe(test_prompt, num_inference_steps=50, guidance_scale=gs).images[0]
    filename = f"outputs/guidance_comparison/gs_{gs}.png"
    image.save(filename)
    display(image)

In [ ]:
# --- Step 6: Parameter exploration — inference steps comparison ---
# More steps = more denoising iterations = generally higher quality, but slower
steps_list = [10, 25, 50]

os.makedirs("outputs/steps_comparison", exist_ok=True)
for steps in steps_list:
    print(f"Generating with num_inference_steps={steps}")
    image = pipe(test_prompt, num_inference_steps=steps, guidance_scale=7.5).images[0]
    filename = f"outputs/steps_comparison/steps_{steps}.png"
    image.save(filename)
    display(image)

In [ ]:
# --- Step 7: Zip and download all outputs ---
import shutil
from google.colab import files

shutil.make_archive("PRODIGY_GA_02_outputs", 'zip', "outputs")
files.download("PRODIGY_GA_02_outputs.zip")

print("\nAll images generated and saved. Download should start automatically.")

## What I Learned

Stable Diffusion generates images through an iterative denoising process: starting
from random noise, a U-Net model progressively removes noise over many steps,
guided by a text prompt encoded via CLIP. This is fundamentally different from
GPT-2's next-token prediction (Task 1) — it's a denoising process operating in
latent (compressed) image space rather than a sequence prediction process.

Key parameters:
- **guidance_scale**: controls how strictly the model follows the prompt
  (low = more creative/random, high = more literal adherence to the prompt)
- **num_inference_steps**: number of denoising iterations
  (more steps generally = higher quality, at the cost of speed)